# Adult Income: Reproducible Machine Learning Analysis

This notebook reimplements in Python the income-classification and segmentation workflow originally built in SAS Viya Model Studio ([SAS project](https://github.com/Omar-Arif-Projects/adult-income-data-mining)). It evaluates the models on the official held-out test set, analyzes fairness by sex and race, and compares the results with SAS Viya.

**Method notes**

- Training uses the official training file only. Tuning uses 5-fold cross-validation within the training set, and the test set is used once, for the final scores.
- Preprocessing follows the SAS pipeline: interval values more than 3 standard deviations from the mean are set to missing, then missing values are imputed (median for numeric variables, most frequent level for categorical variables). `native_country` is dropped, as in SAS (42 levels).
- A single random seed (12345, as in the SAS project) is used throughout.

**Data:** Becker, B. and Kohavi, R. (1996). *Adult*. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/2/adult

In [ ]:
# Install fairlearn if it is not already available (for example, in Google Colab).
try:
    import fairlearn  # noqa: F401
except ImportError:
    %pip install -q fairlearn

In [ ]:
import io, urllib.request, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, silhouette_score)
from sklearn.cluster import KMeans
from fairlearn.metrics import (MetricFrame, selection_rate, true_positive_rate,
                               false_positive_rate, demographic_parity_difference,
                               equalized_odds_difference)

SEED = 12345
np.random.seed(SEED)
Path("figures").mkdir(exist_ok=True)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

def save(name):
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Data

The two files of the UCI Adult dataset (`adult.data` for training, `adult.test` for testing) are downloaded from the UCI Machine Learning Repository and cached in `data/`. If the automatic download fails, download the dataset from https://archive.ics.uci.edu/dataset/2/adult and place both files in `data/`.

In [ ]:
COLS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income"]
UCI_FILES = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/2/adult.zip"
N_TRAIN, N_TEST = 32561, 16281   # row counts of the official files

def _fetch(name):
    # Download one file from UCI, trying the file URL first and the dataset archive second.
    try:
        return urllib.request.urlopen(UCI_FILES + name, timeout=30).read()
    except Exception:
        with urllib.request.urlopen(UCI_ZIP, timeout=60) as resp, zipfile.ZipFile(io.BytesIO(resp.read())) as zf:
            return zf.read(name)

def _csv(path, **kw):
    return pd.read_csv(path, names=COLS, header=None, skipinitialspace=True, na_values="?", **kw)

def load_adult(cache=Path("data")):
    cache.mkdir(exist_ok=True)
    for name in ("adult.data", "adult.test"):
        path = cache / name
        if not path.exists():
            try:
                path.write_bytes(_fetch(name))
            except Exception as err:
                raise RuntimeError(
                    f"Could not download {name} from the UCI repository. Download the dataset from "
                    f"https://archive.ics.uci.edu/dataset/2/adult and place {name} in '{cache}/'."
                ) from err
    train = _csv(cache / "adult.data")
    test = _csv(cache / "adult.test", skiprows=1)      # the first line of adult.test is a comment
    if (len(train), len(test)) != (N_TRAIN, N_TEST):
        raise ValueError(f"Unexpected file sizes: {len(train)} training rows and {len(test)} test rows.")
    return train, test

train_raw, test_raw = load_adult()
print(f"Training rows: {len(train_raw):,} | Test rows: {len(test_raw):,}")

## 2. Cleaning

Text columns are stripped of whitespace, trailing periods are removed from the test labels, the target is encoded as 0/1 (`>50K` = 1), and `native_country` is dropped.

In [ ]:
CAT_ALL = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex"]
NUM = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
DROPPED = ["native_country"]   # rejected in the SAS project (42 levels)

def clean(df):
    df = df.copy()
    for c in CAT_ALL + DROPPED + ["income"]:
        df[c] = df[c].str.strip()
    df["income"] = df["income"].str.rstrip(".")
    y = (df["income"] == ">50K").astype(int).reset_index(drop=True)
    X = df.drop(columns=["income"] + DROPPED).reset_index(drop=True)
    return X, y

X_train, y_train = clean(train_raw)
X_test, y_test = clean(test_raw)

print("Share earning >50K  | train: %.4f   test: %.4f" % (y_train.mean(), y_test.mean()))
print("\nMissing values in the training inputs:")
print(X_train.isna().sum()[lambda s: s > 0].to_string())

### Exploratory view: income by education

In [ ]:
eda = X_train.assign(high=y_train)
order = eda.groupby("education")["education_num"].mean().sort_values().index
rate = eda.groupby("education")["high"].mean().loc[order] * 100

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(rate.index, rate.values, color="#3B7DD8")
ax.set_xlabel("Share earning >50K (%)")
ax.set_title("Income by education level (training set)")
for i, v in enumerate(rate.values):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8)
plt.tight_layout(); save("education_vs_income"); plt.show()

print("Share earning >50K by sex (training set):")
print((eda.groupby("sex")["high"].mean() * 100).round(1).to_string())

## 3. Preprocessing and metrics

`OutlierToNaN` corresponds to the SAS *Replacement* node (values beyond 3 standard deviations become missing), and the imputers correspond to the *Imputation* node.

**Lift** is measured two ways, because the SAS results report "lift at the 2nd decile" and the export does not say which definition was used:

- *cumulative lift at 20% depth*: the share of `>50K` among the top-scored 20% of people, divided by the overall share;
- *decile-2 lift*: the same ratio for the 10%-20% band only.

Trees give many people identical scores, so ties are handled by their expected value rather than by an arbitrary ordering.

In [ ]:
class OutlierToNaN(BaseEstimator, TransformerMixin):
    # Sets values further than n_std standard deviations from the training mean to NaN.
    def __init__(self, n_std=3.0):
        self.n_std = n_std
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.mean_, self.std_ = np.nanmean(X, 0), np.nanstd(X, 0, ddof=1)
        return self
    def transform(self, X):
        X = np.array(X, float, copy=True)
        X[np.abs(X - self.mean_) > self.n_std * self.std_] = np.nan
        return X
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

def make_pre(cat_cols):
    num = Pipeline([("outliers", OutlierToNaN(3.0)), ("impute", SimpleImputer(strategy="median"))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM), ("cat", cat, cat_cols)], verbose_feature_names_out=False)

def _grouped(y, s):
    g = (pd.DataFrame({"y": np.asarray(y, float), "s": np.asarray(s, float)})
           .groupby("s").agg(n=("y", "size"), pos=("y", "sum")).sort_index(ascending=False))
    return g["n"].to_numpy(), g["pos"].to_numpy(), float(np.mean(y))

def _pos_at(n_arr, p_arr, depth):
    # Expected number of positives among the top `depth` share of people (ties split proportionally).
    target = depth * n_arr.sum()
    cn, cp = np.cumsum(n_arr), np.cumsum(p_arr)
    i = int(np.searchsorted(cn, target, side="left"))
    prev_n = cn[i - 1] if i > 0 else 0.0
    prev_p = cp[i - 1] if i > 0 else 0.0
    return prev_p + p_arr[i] * (target - prev_n) / n_arr[i]

def cum_lift(y, score, depth=0.2):
    # Cumulative lift: top `depth` share of people vs everyone.
    n_arr, p_arr, base = _grouped(y, score)
    return _pos_at(n_arr, p_arr, depth) / (depth * n_arr.sum()) / base

def decile_lift(y, score, decile=2):
    # Lift of a single decile band only (for decile 2: people ranked in the 10%-20% band).
    n_arr, p_arr, base = _grouped(y, score)
    pos = _pos_at(n_arr, p_arr, decile / 10) - (_pos_at(n_arr, p_arr, (decile - 1) / 10) if decile > 1 else 0.0)
    return pos / (0.1 * n_arr.sum()) / base

def evaluate(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {"accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred),
            "roc_auc": roc_auc_score(y, proba), "lift_20": cum_lift(y, proba, 0.2),
            "lift_decile2": decile_lift(y, proba, 2)}

## 4. Decision trees

Four configurations, matching the binary-split SAS trees: two split criteria (Entropy, Gini) times two minimum leaf sizes (6% and 3.5% of the data, i.e. about 1,954 and 1,140 training rows). Maximum depth is 10. The pruning strength (`ccp_alpha`) is chosen per configuration by 5-fold cross-validated accuracy on the training set; if two values tie, the simpler tree wins.

The names follow the SAS convention `CRITERION_leafsize_branches`, so `ENTROPY_6_2` is Entropy, 6% leaves, 2 branches.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
LEAF = {"6": 0.06, "35": 0.035}
ALPHAS = [0.002, 0.001, 0.0005, 0.0]      # simplest first, so ties resolve to the simpler tree

tree_models, rows = {}, []
for crit in ["entropy", "gini"]:
    for code_, frac in LEAF.items():
        pipe = Pipeline([("pre", make_pre(CAT_ALL)),
                         ("clf", DecisionTreeClassifier(criterion=crit, min_samples_leaf=frac,
                                                        max_depth=10, random_state=SEED))])
        gs = GridSearchCV(pipe, {"clf__ccp_alpha": ALPHAS}, cv=cv, scoring="accuracy", n_jobs=1)
        gs.fit(X_train, y_train)
        name = f"{crit.upper()}_{code_}_2"
        tree_models[name] = gs.best_estimator_
        m_tr, m_te = evaluate(gs.best_estimator_, X_train, y_train), evaluate(gs.best_estimator_, X_test, y_test)
        rows.append({"tree": name, "ccp_alpha": gs.best_params_["clf__ccp_alpha"],
                     "leaves": gs.best_estimator_["clf"].get_n_leaves(),
                     "cv_accuracy": gs.best_score_, "test_accuracy": m_te["accuracy"],
                     "train_lift_20": m_tr["lift_20"], "test_lift_20": m_te["lift_20"],
                     "test_lift_decile2": m_te["lift_decile2"]})

tree_table = pd.DataFrame(rows).set_index("tree")
display(tree_table.round(4))

BEST = tree_table["cv_accuracy"].idxmax()
print("Selected by cross-validated accuracy:", BEST)

In [ ]:
best = tree_models[BEST]
print(f"Selected tree {BEST}: {best['clf'].get_n_leaves()} leaves, depth {best['clf'].get_depth()}")

# The selected tree is too large to plot legibly. The rules and figure below show a 4-leaf Entropy
# tree (6% minimum leaf size), the same size as the SAS ENTROPY_6_2 tree.
small = Pipeline([("pre", make_pre(CAT_ALL)),
                  ("clf", DecisionTreeClassifier(criterion="entropy", min_samples_leaf=0.06, max_depth=10,
                                                 max_leaf_nodes=4, random_state=SEED))]).fit(X_train, y_train)
names = small["pre"].get_feature_names_out()
print(f"\nSmall tree: {small['clf'].get_n_leaves()} leaves | test accuracy {accuracy_score(y_test, small.predict(X_test)):.4f}\n")
print(export_text(small["clf"], feature_names=list(names)))

fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(small["clf"], feature_names=list(names), class_names=["<=50K", ">50K"], filled=True,
          rounded=True, impurity=False, proportion=True, fontsize=11, ax=ax)
ax.set_title("Small interpretable tree (Entropy, 4 leaves)")
save("small_tree"); plt.show()

## 5. Independent evaluation on the official test set

The four trees are compared with three reference models: a majority-class baseline, logistic regression, and gradient boosting (untuned, default settings). The reference models show how far a small, interpretable tree is from a stronger model.

In [ ]:
refs = {
    "Majority-class baseline": Pipeline([("pre", make_pre(CAT_ALL)), ("clf", DummyClassifier(strategy="most_frequent"))]),
    "Logistic regression": Pipeline([("pre", make_pre(CAT_ALL)), ("scale", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "Gradient boosting": Pipeline([("pre", make_pre(CAT_ALL)), ("clf", HistGradientBoostingClassifier(random_state=SEED))]),
}
for m in refs.values():
    m.fit(X_train, y_train)

all_models = {**refs, **tree_models}
test_table = pd.DataFrame({k: evaluate(m, X_test, y_test) for k, m in all_models.items()}).T
display(test_table.round(4))

In [ ]:
depths = np.linspace(0.05, 1.0, 20)
def curve(model, X, y):
    p = model.predict_proba(X)[:, 1]
    return [cum_lift(y, p, d) for d in depths]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths * 100, curve(best, X_train, y_train), label=f"{BEST} (train)", color="#3B7DD8", ls="--")
ax.plot(depths * 100, curve(best, X_test, y_test), label=f"{BEST} (test)", color="#3B7DD8")
ax.plot(depths * 100, curve(refs["Gradient boosting"], X_test, y_test), label="Gradient boosting (test)", color="#2E9E6B")
ax.axhline(1, color="grey", lw=1)
ax.set_xlabel("Population depth (%)"); ax.set_ylabel("Cumulative lift")
ax.set_title("Cumulative lift curves"); ax.legend()
save("lift_curves"); plt.show()

## 6. Comparison with SAS Viya

The SAS values are taken from the [SAS project's pipeline summary](https://github.com/Omar-Arif-Projects/adult-income-data-mining/blob/main/pipelines/pipeline_summary.md). Read the comparison with these differences in mind:

- SAS scored trees on its own train/validation/test partitions of the 32,561 training rows. Python is scored on the separate official test file.
- SAS accuracy values are rounded to two decimals in the original results table.
- The SAS lift definition (cumulative or single decile) is not recorded in the export, so both Python definitions are shown.
- Pruning, missing-value handling and tie handling are implemented differently in the two tools.

In [ ]:
sas = pd.DataFrame({
    "SAS accuracy": {"ENTROPY_6_2": 0.82, "ENTROPY_35_2": 0.83, "GINI_6_2": 0.82, "GINI_35_2": 0.83},
    "SAS leaves":   {"ENTROPY_6_2": 4,    "ENTROPY_35_2": 8,    "GINI_6_2": 4,    "GINI_35_2": 8},
    "SAS lift (2nd decile)": {"ENTROPY_6_2": 1.38, "ENTROPY_35_2": 1.16, "GINI_6_2": 1.38, "GINI_35_2": 1.16},
})
py = tree_table.loc[sas.index, ["test_accuracy", "leaves", "test_lift_decile2", "test_lift_20"]]
py.columns = ["Python accuracy", "Python leaves", "Python lift (decile 2 only)", "Python lift (cumulative 20%)"]
cmp_table = pd.concat([sas, py], axis=1)[["SAS accuracy", "Python accuracy", "SAS leaves", "Python leaves",
                                          "SAS lift (2nd decile)", "Python lift (decile 2 only)", "Python lift (cumulative 20%)"]]
print("A. Python trees pruned by cross-validation:")
display(cmp_table.round(4))

The Python trees above are much larger than the SAS ones (8 to 23 leaves against 4 to 8), and a tree with few leaves has few distinct scores, which changes lift. To compare like with like, the trees below are grown with the **same leaf count as the SAS tree** (same criterion and minimum leaf size, `max_leaf_nodes` set to the SAS leaf count).

In [ ]:
prows = {}
for name in sas.index:
    crit, code_, _ = name.split("_")
    clf = DecisionTreeClassifier(criterion=crit.lower(), min_samples_leaf=LEAF[code_], max_depth=10,
                                 max_leaf_nodes=int(sas.loc[name, "SAS leaves"]), random_state=SEED)
    m = Pipeline([("pre", make_pre(CAT_ALL)), ("clf", clf)]).fit(X_train, y_train)
    ev = evaluate(m, X_test, y_test)
    prows[name] = {"Python accuracy": ev["accuracy"], "Python leaves": m["clf"].get_n_leaves(),
                   "Python lift (decile 2 only)": ev["lift_decile2"], "Python lift (cumulative 20%)": ev["lift_20"]}
parity = pd.concat([sas, pd.DataFrame(prows).T], axis=1)[["SAS accuracy", "Python accuracy", "SAS leaves", "Python leaves",
                                                          "SAS lift (2nd decile)", "Python lift (decile 2 only)", "Python lift (cumulative 20%)"]]
print("B. Python trees grown to the same number of leaves as the SAS trees:")
display(parity.round(4))

**Reading the comparison**

- **Accuracy agrees closely.** At the same leaf count, Python's test accuracy is within about one percentage point of the SAS figures (0.820 to 0.822 against 0.82 to 0.83).
- **Tree structure differs.** Left to prune itself, scikit-learn keeps more leaves than SAS did (8 to 23 against 4 to 8) for essentially the same accuracy.
- **Lift does not reproduce.** The SAS lift values (1.16 to 1.38) are well below every Python value under either definition, even for trees of the same size. The export does not record how SAS computed them (partition, band or tie handling), so the lift columns should not be read as agreement or disagreement between the tools. Accuracy and tree size are the reliable comparison.

## 7. Segmentation with k-means

The SAS export records the clustering settings but not which input variables were used, so two Python versions are run:

- **numeric only:** the six numeric variables, outlier-cleaned, imputed and standardized;
- **all inputs:** the same numeric variables plus the categorical variables as 0/1 indicators (the SAS cluster nodes include a nominal-distance setting, so categorical inputs may have been used).

Both are clustered on the training data with 2 to 5 clusters (random initialization, similar to the SAS Forgy method; 10 restarts are used here for stability, so the sizes need not match SAS exactly). A cluster holding under 10% of the data is flagged as an outlier cluster, as in the SAS project.

In [ ]:
num_pre = Pipeline([("outliers", OutlierToNaN(3.0)), ("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
Z_num = num_pre.fit_transform(X_train[NUM])
cat_pre = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
Z_all = np.hstack([Z_num, cat_pre.fit_transform(X_train[CAT_ALL])])
FEATURE_SETS = {"numeric only": Z_num, "all inputs": Z_all}

rng = np.random.default_rng(SEED)
sample = rng.choice(len(Z_num), 10000, replace=False)
SAS_SIZES = {2: [16930, 15631], 3: [12015, 7122, 13424], 4: [1261, 12634, 6888, 11778], 5: [1252, 3666, 6128, 13129, 8386]}

labels = {}
for fname, Z in FEATURE_SETS.items():
    krows = []
    for k in [2, 3, 4, 5]:
        km = KMeans(n_clusters=k, init="random", n_init=10, random_state=SEED).fit(Z)
        labels[(fname, k)] = km.labels_
        sizes = sorted(np.bincount(km.labels_), reverse=True)
        krows.append({"k": k, "python sizes": sizes, "SAS sizes": sorted(SAS_SIZES[k], reverse=True),
                      "smallest cluster %": round(100 * min(sizes) / len(Z), 1),
                      "outlier cluster (<10%)": min(sizes) < 0.10 * len(Z),
                      "silhouette": round(silhouette_score(Z[sample], km.labels_[sample]), 3)})
    print(f"K-means on: {fname} ({Z.shape[1]} columns)")
    display(pd.DataFrame(krows).set_index("k"))

In [ ]:
for fname in FEATURE_SETS:
    lab = labels[(fname, 3)]
    prof = (X_train[NUM].assign(cluster=lab, high_income=y_train.to_numpy())
            .groupby("cluster").agg(size=("age", "size"), age=("age", "mean"), education_num=("education_num", "mean"),
                                    hours_per_week=("hours_per_week", "mean"), capital_gain=("capital_gain", "mean"),
                                    share_over_50K=("high_income", "mean")))
    print(f"Profile of the 3 clusters ({fname}):")
    display(prof.round(2))

## 8. Fairness analysis

Two models are checked on the official test set: the selected decision tree and gradient boosting. Each is trained twice, **with** and **without** the `sex` input, to test whether simply removing the attribute removes the gap (the variable `relationship`, for example, can act as a proxy).

Metrics, by group:

- **Selection rate:** the share predicted `>50K`.
- **True positive rate (TPR):** among people who really earn `>50K`, the share predicted `>50K`.
- **False positive rate (FPR):** among people who earn `<=50K`, the share predicted `>50K`.
- **Demographic parity difference:** the largest gap in selection rate between groups.
- **Equalized odds difference:** the larger of the largest TPR gap and the largest FPR gap between groups.

The dataset is from 1994 and records sex as binary only. These numbers describe what a model learns from that data; they are not a statement about the people it represents.

In [ ]:
CAT_NO_SEX = [c for c in CAT_ALL if c != "sex"]
bp = best["clf"].get_params()
def make_model(kind, cats):
    if kind == "Tree":
        clf = DecisionTreeClassifier(criterion=bp["criterion"], min_samples_leaf=bp["min_samples_leaf"],
                                     max_depth=10, ccp_alpha=bp["ccp_alpha"], random_state=SEED)
    else:
        clf = HistGradientBoostingClassifier(random_state=SEED)
    return Pipeline([("pre", make_pre(cats)), ("clf", clf)]).fit(X_train, y_train)

fair_models = {
    f"Tree ({BEST}), with sex": make_model("Tree", CAT_ALL),
    f"Tree ({BEST}), without sex": make_model("Tree", CAT_NO_SEX),
    "Gradient boosting, with sex": make_model("GB", CAT_ALL),
    "Gradient boosting, without sex": make_model("GB", CAT_NO_SEX),
}
preds = {k: m.predict(X_test) for k, m in fair_models.items()}

# Confirm that `sex` is absent from the second model's inputs and compare the two trees' predictions.
t_with, t_without = fair_models[f"Tree ({BEST}), with sex"], fair_models[f"Tree ({BEST}), without sex"]
used = lambda m: {n for n in np.asarray(m["pre"].get_feature_names_out())[m["clf"].tree_.feature[m["clf"].tree_.feature >= 0]]}
print("Sex columns in the without-sex tree's inputs:", [n for n in t_without["pre"].get_feature_names_out() if n.startswith("sex")])
print("Variables the with-sex tree splits on:", sorted({n.split("_")[0] for n in used(t_with)}))
print("Share of test rows with identical predictions (tree with vs without sex): %.4f" %
      (preds[f"Tree ({BEST}), with sex"] == preds[f"Tree ({BEST}), without sex"]).mean())

print("Test-set group sizes and actual >50K rates")
for attr in ["sex", "race"]:
    g = X_test.assign(high=y_test).groupby(attr)["high"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "actual_rate"})
    display(g.round(3))

In [ ]:
METRICS = {"selection_rate": selection_rate, "tpr": true_positive_rate, "fpr": false_positive_rate, "accuracy": accuracy_score}

summary = []
for name, pred in preds.items():
    for attr in ["sex", "race"]:
        summary.append({"model": name, "attribute": attr, "accuracy": accuracy_score(y_test, pred),
                        "demographic_parity_diff": demographic_parity_difference(y_test, pred, sensitive_features=X_test[attr]),
                        "equalized_odds_diff": equalized_odds_difference(y_test, pred, sensitive_features=X_test[attr])})
display(pd.DataFrame(summary).round(3))

by_sex = {name: MetricFrame(metrics=METRICS, y_true=y_test, y_pred=pred, sensitive_features=X_test["sex"]).by_group
          for name, pred in preds.items()}
print("Group metrics by sex:")
display(pd.concat(by_sex, names=["model"]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
labels = list(preds.keys())
x = np.arange(len(labels)); w = 0.38
for ax, metric, title in zip(axes, ["selection_rate", "tpr"], ["Selection rate (predicted >50K)", "True positive rate"]):
    f_vals = [by_sex[l].loc["Female", metric] for l in labels]
    m_vals = [by_sex[l].loc["Male", metric] for l in labels]
    ax.bar(x - w / 2, f_vals, w, label="Female", color="#E08A3C")
    ax.bar(x + w / 2, m_vals, w, label="Male", color="#3B7DD8")
    ax.set_xticks(x); ax.set_xticklabels([l.replace(", ", "\n") for l in labels], fontsize=8)
    ax.set_ylim(0, max(f_vals + m_vals) * 1.25)
    ax.set_title(title); ax.legend(loc="upper center", ncol=2, frameon=False)
plt.tight_layout(); save("fairness_by_sex"); plt.show()

## 9. Summary of findings

**Model performance (official test set)**

- A majority-class baseline reaches 76.4% accuracy, so that is the number to beat. Logistic regression reaches 84.5% and untuned gradient boosting 86.1%.
- The decision trees reach 82.0% to 82.3%. The tree selected by cross-validation (`ENTROPY_35_2`, 23 leaves) scores 82.3% accuracy and 0.867 ROC AUC.
- Small trees give up about 2 to 4 percentage points of accuracy against the stronger models in exchange for rules a person can read.

**Segmentation**

- Clustering on all inputs reproduces the SAS two-cluster split almost exactly (16,828 / 15,733 against 16,930 / 15,631). With 3, 4 or 5 clusters it isolates a small cluster of about 1,250 to 1,300 people (about 4% of the data); SAS also had a cluster of about 1,250 people at 4 and 5 clusters (1,261 and 1,252). In the three-cluster profile this group averages a capital gain above 10,000 and 94% of it earns more than $50K.
- The three-cluster solutions differ between the tools. K-means can end in different local optima, and the SAS node was limited to 10 iterations.
- Silhouette scores for the all-inputs clusters are low (about 0.11 to 0.12), so the segments overlap heavily.

**Fairness (test set, by sex)**

- In the test data 30.0% of men and 10.9% of women earn more than $50K.
- The selected tree predicts >50K for 24.6% of men and 6.1% of women: men are flagged about four times as often as women, against about 2.75 times in the underlying data (the absolute gap, 18.5 points against 19.1, is similar). Its true positive rate is 0.54 for men and 0.38 for women, so high-earning women are missed more often.
- Removing `sex` from the tree changed none of its predictions, even though the tree did split on sex. For gradient boosting, removing it narrowed the gaps only slightly (demographic parity difference 0.177 to 0.168).
- Race gaps are noisier because some groups are small (135 and 159 test rows for "Other" and "Amer-Indian-Eskimo"), so the large equalized odds difference for race should be read with caution.

**Limitations**

- The data is from 1994, and sex and race are coded in broad categories. The results describe patterns in this dataset, not the people it represents.
- No bias mitigation was attempted. This notebook measures gaps and does not claim to fix them.
- The SAS comparison is a cross-check, not an exact replication (see section 6).

## 10. Reproducibility

Library versions used for the run stored in this notebook:

In [ ]:
import sklearn, fairlearn, matplotlib, sys
print("Python", sys.version.split()[0])
for m in (np, pd, sklearn, fairlearn, matplotlib):
    print(m.__name__, m.__version__)